# Regional Rank Stability

Spearman rank correlation between baseline regional vulnerability ranking
and the worst-case perturbed ranking (infrastructure_damage weight -10%).
Each point is one of the 17 Myanmar regions.
The tight clustering around the diagonal confirms the ranking is robust.

Wide figure (10×4.5 in at 400 DPI) for two-column insertion as `figure*`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
from scipy.stats import spearmanr, linregress
import matplotlib.pyplot as plt

from src.processing import extract_health_impacts, classify_hice_type
print('Imports OK')

: 

In [ ]:
DATA_PATH = '../data/myanmar_conflict_clean.csv'
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Data loaded: {len(df)} events')

hice_mask = extract_health_impacts(df)
hice = df[hice_mask].copy()
hice['hice_type'] = classify_hice_type(hice)
print(f'HICE events: {len(hice)}')

In [ ]:
W_BASE = dict(
    personnel_targeting=1.0,
    systemic_attack=0.9,
    infrastructure_damage=0.6,
    access_disruption=0.5,
    humanitarian_disruption=0.3,
)

def compute_rankings(weights):
    h = hice.copy()
    h['w'] = h['hice_type'].map(weights).fillna(0.3)
    h['impact'] = h['w'] * (1 + np.log1p(h['fatalities'].clip(lower=0)))
    v = h.groupby('admin1').agg(Score=('impact', 'sum')).reset_index()
    v = v.sort_values('Score', ascending=False)
    v['Rank'] = range(1, len(v) + 1)
    return v[['admin1', 'Score', 'Rank']]

baseline = compute_rankings(W_BASE)
print('Baseline ranking:')
baseline

In [ ]:
# Find worst-case perturbation
W_CASE = W_BASE.copy()
W_CASE['infrastructure_damage'] = 0.6 * 0.9  # -10%

perturbed = compute_rankings(W_CASE)
merged = baseline.merge(perturbed, on='admin1', suffixes=('_base', '_pert'))

rho, pv = spearmanr(merged['Rank_base'], merged['Rank_pert'])
print(f"infrastructure_damage -10%: Spearman's \u03c1 = {rho:.6f}, p = {pv:.2e}")
merged.sort_values('Rank_base')

In [ ]:
# Scatter plot: baseline rank vs perturbed rank
fig, ax = plt.subplots(1, 1, figsize=(10, 4.5))

x = merged['Rank_base']
y = merged['Rank_pert']

slope, intercept, r_val, p_val, std_err = linregress(x, y)
xx = np.linspace(0.5, 17.5, 100)
ax.plot(xx, slope * xx + intercept, color='#0066cc', linewidth=1.5, linestyle='--')
ax.plot([0, 18], [0, 18], color='#ccc', linewidth=0.8, zorder=0)

for _, row in merged.iterrows():
    ax.annotate(row['admin1'], (row['Rank_base'], row['Rank_pert']),
                fontsize=9, ha='center', va='bottom',
                xytext=(0, 7), textcoords='offset points', color='#1d1d1f')

ax.scatter(x, y, s=55, color='#0066cc', edgecolors='white', linewidth=0.5, zorder=5)

ax.text(0.3, 16.6, f"Spearman's \u03c1 = {rho:.4f}", fontsize=13,
        fontweight='bold', color='#1d1d1f', va='top')
ax.text(0.3, 15.5, 'infrastructure_damage weight \u221210%', fontsize=10,
        color='#666', va='top')

ax.set_xlabel('Baseline Regional Rank', fontsize=12)
ax.set_ylabel('Perturbed Regional Rank', fontsize=12)
ax.set_title('Regional Rank Stability (Worst-Case Perturbation)', fontsize=14, fontweight='bold')
ax.set_xlim(0, 18)
ax.set_ylim(0, 18)
ax.set_xticks(range(1, 18))
ax.set_yticks(range(1, 18))
ax.grid(True, alpha=0.25)
ax.tick_params(labelsize=10)

plt.tight_layout()
plt.savefig('../research/assets/rank_stability.png', dpi=400, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: research/assets/rank_stability.png')